In [ ]:
import pandas as pd

In [ ]:
PC_DATA_PARQUET = "../file_processing/processed/pc_data.parquet"

FINAL_CHOICE_PARQUET = "../file_processing/processed/final_choice.parquet"

NULL_PARQUET = "../file_processing/processed/null_run_pc_matched.parquet"

NULL_FINAL_CHOICE_PARQUET = (
    "../file_processing/processed/null_run_final_choice_matched.parquet")

In [ ]:
final_choice_data = pd.read_parquet(FINAL_CHOICE_PARQUET)

final_choice_data

In [ ]:
pc_data = pd.read_parquet(PC_DATA_PARQUET)

print(pc_data.shape)
print(f"Number of Unique Participants:{pc_data["participant_id"].nunique()}")
pc_data.head()

In [ ]:
null_data = pd.read_parquet(NULL_PARQUET)

print(null_data.shape)
print(f"Number of Unique Participants:{null_data["participant_id"].nunique()}")
null_data.head()

In [ ]:
reject_condition = (pc_data['generation_key'].str.contains('gen_1', regex=False))
gen1_reject_df = pc_data[reject_condition].copy()
print(gen1_reject_df.shape)
print(gen1_reject_df['participant_id'].nunique())
gen1_reject_df.head(10)

In [ ]:
select_condition = (pc_data['generation_key'].str.contains('gen_5', regex=False)) & (pc_data['is_selected_vase'] == True)
gen5_select_df = pc_data[select_condition].copy()
print(gen5_select_df.shape)
print(gen5_select_df['participant_id'].nunique())
gen5_select_df.head(10)

## KDE density difference map

Compare where the two populations sit in PC space:

- **Gen 5, null run** — the baseline. The same participants were offered the same
  gen-1 vases, but every choice was made at random. Five generations of the same
  generator then ran on top of those random choices.
- **Gen 5, real run** — the shapes people actually settled on by the end.

For each plane (PC1/PC2, PC3/PC4 and PC5/PC6) we estimate a Gaussian KDE for each group
on a shared grid and plot `density(real) − density(null)`. Red marks regions humans
chose **more** often than chance would, blue regions they chose **less** often.

**Why this baseline and not gen 1.** Gen-5 vases differ from gen-1 vases for two quite
different reasons: people chose them, and the generator widens the distribution every
generation regardless of what anyone picks. A gen-1 baseline cannot separate the two, so
most of what it shows is the generator expanding. The null run has been through exactly
the same five generations of the same generator, from exactly the same starting vases —
the only difference is that its choices were random. Subtracting it cancels the
generative drift and leaves the effect of *choosing*.

Two choices matter for the difference to be meaningful:

1. **Shared bandwidth.** Both groups are standardised by the same pooled mean/SD and
   given the same scalar bandwidth. If each KDE picked its own bandwidth from its own
   covariance, a difference in *spread* would show up as a difference in *location* —
   an artefact of the smoothing, not the data.
2. **Shared grid and normalisation.** Each KDE integrates to 1 over the plane, so the
   two surfaces are on the same scale and the difference is a genuine redistribution of
   probability mass (it sums to ~0). Here both groups are the same size (13,620 vases,
   2,724 participants, five rounds each) so the comparison is as balanced as it gets.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from pathlib import Path

FIG_DIR = Path("figures/pc_density_diff_map")

# Sequential single hue for the density panels; blue<->red diverging with a
# neutral (not coloured) midpoint for the difference, so "no change" reads as
# nothing rather than as a third category.
SEQ_BLUE = LinearSegmentedColormap.from_list("seq_blue", [
    "#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
    "#256abf", "#184f95", "#0d366b"])
DIVERGING = LinearSegmentedColormap.from_list("blue_red", [
    "#0d366b", "#184f95", "#2a78d6", "#86b6ef", "#cde2fb",
    "#f0efec",
    "#f7c9c8", "#ee9695", "#e34948", "#bd2f2e", "#8f1f1f"])
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"

In [ ]:
# 13 sessions are ingested twice in the parquet — 3,900 byte-identical rows.
# Values are the same either way, but left in they'd double-weight those
# participants in the KDE, so drop them before anything else.
n_before = len(pc_data)
pc_clean = pc_data.drop_duplicates()
print(f"dropped {n_before - len(pc_clean):,} exact duplicate rows "
      f"({n_before:,} -> {len(pc_clean):,})")

# One row per vase, PC1..PC6 as columns. `pivot` (not `pivot_table`) is
# deliberate: it raises on a repeated key rather than silently averaging.
VASE_KEYS = ["participant_id", "session_id", "generation_key", "vase"]


def to_wide(df):
    return df.pivot(index=VASE_KEYS, columns="pc_name",
                    values="pc_value").reset_index()


# endswith rather than contains: 'gen_1' as a substring would also match a
# gen_10+ key if the design ever runs more than nine generations.
is_gen1 = pc_clean["generation_key"].str.endswith("gen_1")
is_gen5 = pc_clean["generation_key"].str.endswith("gen_5")

# Baseline: EVERY gen-1 vase, chosen and passed over alike. This is the
# landscape the task starts from — what the generator put in front of people
# before any choice had been made — so it carries no selection signal of its
# own. (Gen-1 rejects alone would have made the baseline the losing half of a
# choice, which is already a selection-shaped sample.)
gen1_all = to_wide(pc_clean[is_gen1])
gen5_select = to_wide(pc_clean[is_gen5 & pc_clean["is_selected_vase"]])

print(f"gen1 baseline (all vases): {len(gen1_all):,} vases, "
      f"{gen1_all['participant_id'].nunique():,} participants")
print(f"gen5 selected:             {len(gen5_select):,} vases, "
      f"{gen5_select['participant_id'].nunique():,} participants")
gen1_all.head()

In [ ]:
# The "king of kings" vase: after five rounds each participant was shown the
# five vases they had settled on (one per round) and picked the most beautiful.
# final_choice.parquet records only WHICH round won, so the PCs have to come
# from pc_data — the winner is that round's gen-5 selected vase.
#
# Rounds are 1-indexed there ("Round 1") and 0-indexed in pc_data
# ("round_0_gen_5"), so the mapping subtracts one.
round_n = final_choice_data["selected_round"].str.extract(r"(\d+)")[0].astype(int)
fc_keys = final_choice_data.assign(
    generation_key="round_" + (round_n - 1).astype(str) + "_gen_5"
)[["participant_id", "session_id", "generation_key"]]

# validate="1:1" so a silent fan-out can't inflate the group, and an inner
# merge would hide misses — check them explicitly instead.
final_choice = fc_keys.merge(
    gen5_select, on=["participant_id", "session_id", "generation_key"],
    how="left", validate="1:1",
)
missing = final_choice["PC1"].isna().sum()
assert missing == 0, f"{missing} final choices did not match a gen-5 selected vase"

print(f"final choice: {len(final_choice):,} vases, "
      f"{final_choice['participant_id'].nunique():,} participants "
      f"(one per participant)")
print("\nwhich round won:")
print(final_choice_data["selected_round"].value_counts().sort_index().to_string())

In [ ]:
# The null runs: the SAME gen-1 vases offered to the SAME participants, but
# every selection made at random. There are now 10 independent repeats.
#
# POOL the repeats, do NOT average their PC values. Concatenating the vases
# averages the ten density surfaces while keeping every vase where it actually
# landed. Averaging the PC values instead would shrink the run-to-run variance
# by 1/10 and produce a spuriously tight null cloud — which would invert the
# finding that random selection spreads WIDER than human selection.
assert null_data["session_id"].isna().all(), "null run unexpectedly has session ids"
NULL_VASE_KEYS = ["null_repeat", "participant_id", "generation_key", "vase"]
assert not null_data.duplicated(NULL_VASE_KEYS + ["pc_name"]).any()
REPEATS = sorted(null_data["null_repeat"].unique())

null_gen5_all = null_data[
    null_data["generation_key"].str.endswith("gen_5") & null_data["is_selected_vase"]
].pivot(index=NULL_VASE_KEYS, columns="pc_name", values="pc_value").reset_index()
null_gen5_select = null_gen5_all  # pooled across every repeat

# Verify the runs really are "pc matched": gen 1 must be the same vases in
# every repeat and in the real run, with only the choice differing.
_m = ["participant_id", "generation_key", "vase", "pc_name"]
_cols = _m + ["pc_value", "is_selected_vase"]
_r1 = pc_clean.loc[pc_clean["generation_key"].str.endswith("gen_1"), _cols]
_agree = []
for _rep, _grp in null_data[null_data["generation_key"].str.endswith("gen_1")].groupby("null_repeat"):
    _j = _r1.merge(_grp[_cols], on=_m, suffixes=("_real", "_null"))
    assert len(_j) == len(_r1), f"repeat {_rep}: gen-1 vases do not line up"
    assert np.allclose(_j["pc_value_real"], _j["pc_value_null"]), f"repeat {_rep}: gen-1 PCs differ"
    _agree.append((_j["is_selected_vase_real"] == _j["is_selected_vase_null"]).mean())

print(f"{len(REPEATS)} null repeats; gen-1 offerings identical to the real run in all of them")
print(f"gen-1 choices agreeing with the real run: "
      f"{min(_agree):.1%}–{max(_agree):.1%} across repeats (~50% = chance)")


# The exchangeable unit is the LINEAGE, not the participant: each round starts
# from its own gen-1 pair, so a round-0 vase and a round-3 vase are not
# interchangeable even under the null. Blocking on (participant, round) pairs
# each real vase with the 10 null vases grown from the same starting shapes.
def lineage_block(df):
    return (df["participant_id"] + "|"
            + df["generation_key"].str.extract(r"(round_\d+)")[0])


gen5_select["block"] = lineage_block(gen5_select)
null_gen5_select["block"] = lineage_block(null_gen5_select)

print()
print(f"real gen5 selected: {len(gen5_select):,} vases")
print(f"null gen5 selected: {len(null_gen5_select):,} vases "
      f"({len(REPEATS)} repeats pooled)")
print(f"lineage blocks: {gen5_select['block'].nunique():,} "
      f"(1 real + {len(REPEATS)} null vases each)")


In [ ]:
# The null "king of kings" choice: for each repeat, one winning round per
# participant, resolved to that repeat's own gen-5 selected vase.
null_final_choice_data = pd.read_parquet(NULL_FINAL_CHOICE_PARQUET)

_nfc_round = null_final_choice_data["selected_round"].str.extract(r"(\d+)")[0].astype(int)
_nfc_keys = null_final_choice_data.assign(
    generation_key="round_" + (_nfc_round - 1).astype(str) + "_gen_5"
)[["null_repeat", "participant_id", "generation_key"]]

null_final_choice = _nfc_keys.merge(
    null_gen5_select, on=["null_repeat", "participant_id", "generation_key"],
    how="left", validate="1:1",
)
assert null_final_choice["PC1"].notna().all(), "a null final choice did not resolve"

# Here the block is the PARTICIPANT, not the lineage: which round wins is part
# of what the final-choice step decides, so real and null pick different rounds
# and there is no round to match on.
final_choice["block"] = final_choice["participant_id"]
null_final_choice["block"] = null_final_choice["participant_id"]

print(f"real final choice: {len(final_choice):,} vases")
print(f"null final choice: {len(null_final_choice):,} vases "
      f"({len(REPEATS)} repeats pooled)")
print()
print("which round won (% of choices) — real vs null:")
_rr = final_choice_data["selected_round"].value_counts(normalize=True).sort_index()
_nr = null_final_choice_data["selected_round"].value_counts(normalize=True).sort_index()
for _k in _rr.index:
    print(f"  {_k}: real {_rr[_k]:5.1%}   null {_nr[_k]:5.1%}")


In [ ]:
# The gen-1 baseline is not just a different shape from the later vases — it
# lives in a much smaller box. Gen 1 looks like it's drawn from a bounded range
# (~±8 on every PC); five generations of variation carry the population well
# past it. This is why the difference maps have a rim of "more vases than
# baseline" around the edge: out there the baseline is genuinely empty, not
# merely sparse.
PCS = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6"]
COMPARISONS = {"gen5 real": gen5_select, "gen5 null": null_gen5_select,
               "fc real": final_choice, "fc null": null_final_choice}

for pc in PCS:
    ranges = "   ".join(
        f"{name} [{df[pc].min():6.2f}, {df[pc].max():5.2f}]"
        for name, df in COMPARISONS.items())
    print(f"{pc}: gen1 [{gen1_all[pc].min():6.2f}, {gen1_all[pc].max():5.2f}]   {ranges}")

print()
for name, df in COMPARISONS.items():
    outside = np.zeros(len(df), dtype=bool)
    for pc in PCS:
        outside |= (df[pc] < gen1_all[pc].min()) | (df[pc] > gen1_all[pc].max())
    print(f"{name} vases outside the gen-1 {PCS[0]}–{PCS[-1]} box: "
          f"{outside.sum():,} / {len(df):,} ({outside.mean():.1%})")

In [ ]:
from scipy.ndimage import gaussian_filter


def binned_kde(xy, edges_x, edges_y, sigma_px, cell_area):
    """KDE by histogram + Gaussian convolution — the same estimator as
    `kde_grid` up to the binning, but ~600x faster because cost stops scaling
    with the number of points. `sigma_px` is the kernel SD in grid cells,
    (row, col) order. Normalised to integrate to 1."""
    H, _, _ = np.histogram2d(xy[:, 1], xy[:, 0], bins=[edges_y, edges_x])
    dens = gaussian_filter(H, sigma=sigma_px, mode="constant")
    return dens / (dens.sum() * cell_area)


def grid_spec(res_plane, bw, sd):
    """Bin edges and kernel width matching a plane's grid."""
    gx, gy = res_plane["gx"], res_plane["gy"]
    dx, dy = gx[0, 1] - gx[0, 0], gy[1, 0] - gy[0, 0]
    edges_x = np.concatenate([gx[0] - dx / 2, [gx[0, -1] + dx / 2]])
    edges_y = np.concatenate([gy[:, 0] - dy / 2, [gy[-1, 0] + dy / 2]])
    return edges_x, edges_y, (bw * sd[1] / dy, bw * sd[0] / dx), dx * dy


def plane_limits(a, b, pad=0.06):
    """Window covering the 0.5–99.5 percentile range of *both* groups.

    The envelope rather than a pooled percentile: the gen-1 baseline sits in a
    far smaller box than gen-5 (see the escape check above), and a pooled
    percentile — dominated by the larger baseline sample — would crop the gen-5
    tails out of the picture entirely.
    """
    lims = []
    for i in (0, 1):
        lo = min(np.percentile(a[:, i], 0.5), np.percentile(b[:, i], 0.5))
        hi = max(np.percentile(a[:, i], 99.5), np.percentile(b[:, i], 99.5))
        m = (hi - lo) * pad
        lims.append((lo - m, hi + m))
    return lims


def kde_grid(xy, mu, sd, bw, gx, gy):
    """Gaussian KDE on a grid, smoothed identically across groups.

    The points are standardised by the *pooled* mu/sd and given the same scalar
    bandwidth, so both groups get the same smoothing kernel. Left to itself,
    gaussian_kde scales its kernel by each group's own covariance — which would
    turn a difference in spread into an apparent difference in location.

    Returns density in the original PC units (integrates to 1 over the plane).
    """
    k = gaussian_kde(((xy - mu) / sd).T, bw_method=bw)
    gz = np.column_stack([(gx.ravel() - mu[0]) / sd[0],
                          (gy.ravel() - mu[1]) / sd[1]])
    return k(gz.T).reshape(gx.shape) / (sd[0] * sd[1])

In [ ]:
PLANES = [("PC1", "PC2"), ("PC3", "PC4"), ("PC5", "PC6")]
GRID_N = 220


KDE_MODE = "binned"   # "binned" (fast) or "exact" (scipy gaussian_kde)


def compute_planes(a_df, b_df, planes=PLANES, grid_n=GRID_N, mode=None):
    """KDE surfaces and their difference per plane, baseline A vs group B.

    `diff` is B − A throughout, so red means "more than baseline".

    mode:
      "binned" — histogram + Gaussian convolution. Default: the exact
                 estimator costs ~5 s per surface and scales with the number
                 of points, which is untenable now the baseline is 136,200
                 pooled null vases. Agreement with exact is checked below.
      "exact"  — scipy gaussian_kde, kept for that check.
      "grid"   — geometry only, no densities. For callers that just need the
                 grid and bandwidth (the permutation tests rebuild their own
                 surfaces), so computing densities here would be wasted work.
    """
    mode = mode or KDE_MODE
    out = {}
    for xk, yk in planes:
        a = a_df[[xk, yk]].to_numpy()
        b = b_df[[xk, yk]].to_numpy()
        pooled = np.vstack([a, b])
        mu, sd = pooled.mean(0), pooled.std(0)
        bw = pooled.shape[0] ** (-1 / 6)  # Scott's rule, d=2, from the pooled n

        (x0, x1), (y0, y1) = plane_limits(a, b)
        gx, gy = np.meshgrid(np.linspace(x0, x1, grid_n),
                             np.linspace(y0, y1, grid_n))
        cell_area = (gx[0, 1] - gx[0, 0]) * (gy[1, 0] - gy[0, 0])
        if mode == "grid":
            out[(xk, yk)] = dict(gx=gx, gy=gy, d1=None, d5=None, diff=None,
                                 extent=(x0, x1, y0, y1), bw=bw,
                                 cell_area=cell_area, mass_1=np.nan,
                                 mass_5=np.nan, centroid_1=a.mean(0),
                                 centroid_5=b.mean(0))
            continue
        if mode == "exact":
            d1 = kde_grid(a, mu, sd, bw, gx, gy)
            d5 = kde_grid(b, mu, sd, bw, gx, gy)
        else:
            _spec = dict(gx=gx, gy=gy)
            ex, ey, sig_px, _c = grid_spec(_spec, bw, sd)
            d1 = binned_kde(a, ex, ey, sig_px, cell_area)
            d5 = binned_kde(b, ex, ey, sig_px, cell_area)
        out[(xk, yk)] = dict(
            gx=gx, gy=gy, d1=d1, d5=d5, diff=d5 - d1, extent=(x0, x1, y0, y1),
            bw=bw, cell_area=cell_area,
            mass_1=d1.sum() * cell_area, mass_5=d5.sum() * cell_area,
            centroid_1=a.mean(0), centroid_5=b.mean(0),
        )
    return out


def report_planes(res, b_name):
    for (xk, yk), r in res.items():
        dx, dy = r["centroid_5"] - r["centroid_1"]
        # mass captured on the grid should be ~1.0; well below that means the
        # window is clipping the tails and the difference is missing some
        print(f"{xk}/{yk}: bw={r['bw']:.4f}  mass on grid: "
              f"baseline={r['mass_1']:.3f} {b_name}={r['mass_5']:.3f}  "
              f"centroid shift: {xk} {dx:+.2f}, {yk} {dy:+.2f}")


# How much of the shape variance each plane actually carries. PC5/PC6 is the
# smallest slice by some way, so a strong-looking difference there still means
# a smaller change in the vase than the same-looking difference in PC1/PC2.
_pc_var = gen1_all[[c for p in PLANES for c in p]].var()
print("share of gen-1 variance per plane:")
for xk, yk in PLANES:
    print(f"  {xk}/{yk}: {(_pc_var[xk] + _pc_var[yk]) / _pc_var.sum():.1%}")
print()

# Baseline A = the null run (random choices), comparison B = the real run.
# diff = real - null, so red marks shapes humans chose MORE than chance.
results = compute_planes(null_gen5_select, gen5_select)
report_planes(results, "real")

In [ ]:
def _style(ax, xk, yk, title, extent):
    ax.axhline(0, color=MUTED, lw=0.6, alpha=0.5, zorder=1)
    ax.axvline(0, color=MUTED, lw=0.6, alpha=0.5, zorder=1)
    ax.set_title(title, fontsize=10, color=INK, pad=8)
    ax.set_xlabel(xk, fontsize=9, color=MUTED)
    ax.set_ylabel(yk, fontsize=9, color=MUTED)
    ax.tick_params(labelsize=8, colors=MUTED, length=2)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_aspect("equal")  # PCs share units, so don't distort the plane
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.set_facecolor(SURFACE)


def _bar(fig, im, ax, label):
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.ax.tick_params(labelsize=7, colors=MUTED, length=2)
    cb.outline.set_visible(False)
    cb.set_label(label, fontsize=8, color=MUTED)


fig, axes = plt.subplots(len(PLANES), 3, figsize=(16, 5.25 * len(PLANES)),
                         facecolor=SURFACE)
for row, (xk, yk) in enumerate(PLANES):
    r = results[(xk, yk)]
    vmax = max(r["d1"].max(), r["d5"].max())
    dmax = np.abs(r["diff"]).max()

    # both density panels share a scale, so they share a single colorbar
    for col, (title, Z) in enumerate([("Gen 5 — null run (random)", r["d1"]),
                                      ("Gen 5 — real (human choice)", r["d5"])]):
        ax = axes[row, col]
        im = ax.imshow(Z, origin="lower", extent=r["extent"], cmap=SEQ_BLUE,
                       vmin=0, vmax=vmax, aspect="equal", zorder=0)
        ax.contour(r["gx"], r["gy"], Z, levels=5, colors=INK,
                   linewidths=0.4, alpha=0.22, zorder=2)
        _style(ax, xk, yk, title, r["extent"])
    _bar(fig, im, axes[row, 1], "density")

    ax = axes[row, 2]
    im = ax.imshow(r["diff"], origin="lower", extent=r["extent"], cmap=DIVERGING,
                   vmin=-dmax, vmax=dmax, aspect="equal", zorder=0)
    ax.contour(r["gx"], r["gy"], r["diff"], levels=[0], colors=INK,
               linewidths=0.7, alpha=0.5, zorder=2)
    ax.contour(r["gx"], r["gy"], r["diff"], levels=[-0.5 * dmax, 0.5 * dmax],
               colors=INK, linewidths=0.4, alpha=0.3, linestyles="--", zorder=2)

    c1, c5 = r["centroid_1"], r["centroid_5"]
    ax.annotate("", xy=c5, xytext=c1, zorder=3,
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.4,
                                shrinkA=0, shrinkB=0))
    ax.scatter(*c1, s=26, facecolor="none", edgecolor=INK, lw=1.2, zorder=4)
    ax.scatter(*c5, s=26, color=INK, zorder=4)
    _style(ax, xk, yk, "Difference (real − null)", r["extent"])
    _bar(fig, im, ax, "Δ density")

axes[0, 2].text(0.02, 0.02,
                "○ → ● centroid shift\nred = more vases than baseline   "
                "blue = fewer",
                transform=axes[0, 2].transAxes, fontsize=7.5, color=MUTED,
                va="bottom", ha="left")

fig.suptitle("What human selection adds over random selection\n"
             "Gen 5 vases, same generator and same participants; "
             f"n={len(null_gen5_select):,} null (random) vs "
             f"{len(gen5_select):,} real (human)",
             fontsize=12, color=INK, y=0.98)
fig.tight_layout(rect=(0, 0, 1, 0.945))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "pc_density_diff_kde.png", dpi=200, facecolor=SURFACE)
plt.show()

### The difference map on its own

The same difference surface as the right-hand column above, at full size and without the
centroid markers or the two component densities — just `density(real) − density(null)`
for each plane. Each panel is scaled symmetrically about zero so the neutral midpoint
means "no change", and the panels keep separate scales because density units differ
between them.

The two outlines mark where each run's vases actually live: the dashed contour encloses
95% of the null run's gen-5 selections, the solid green one 95% of the real run's. Unlike
the old gen-1 comparison these two contours sit almost on top of each other — both runs
expanded through the same generator, so the *extent* of the cloud is no longer the story.
What is left is the redistribution of mass **inside** the shared region, which is the
part attributable to choosing.


In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.lines import Line2D

# Outline enclosing this share of each group's own vases. 0.95 traces the bulk
# of a population without chasing the handful of extreme outliers that a 100%
# boundary would stretch to.
BOUND_MASS = 0.95
GEN1_COLOR, GEN5_COLOR = "#0b0b0b", "#008300"


def hdr_level(d, cell_area, mass=BOUND_MASS):
    """Iso-density level enclosing `mass` of this KDE — the contour that bounds
    the smallest region holding that share of the group's vases."""
    flat = np.sort(d.ravel())[::-1]
    cumulative = np.cumsum(flat) * cell_area
    return flat[min(np.searchsorted(cumulative, mass), flat.size - 1)]


def plot_diff_planes(res, b_label, filename, a_label="Gen 1 baseline"):
    # width scales with the panel count so adding a plane does not squash them
    fig, axes = plt.subplots(1, len(res), figsize=(8.0 * len(res), 8),
                             facecolor=SURFACE)

    for ax, (xk, yk) in zip(np.atleast_1d(axes), res):
        r = res[(xk, yk)]
        # symmetric about zero per plane, so the neutral midpoint really is
        # "no change"; scales are per-plane because density units differ
        dmax = np.abs(r["diff"]).max()

        im = ax.imshow(r["diff"], origin="lower", extent=r["extent"],
                       cmap=DIVERGING, vmin=-dmax, vmax=dmax, aspect="equal",
                       zorder=0)

        # where each group's vases actually live: dashed = the starting
        # region, solid = where they ended up
        ax.contour(r["gx"], r["gy"], r["d1"],
                   levels=[hdr_level(r["d1"], r["cell_area"])],
                   colors=GEN1_COLOR, linewidths=1.6, linestyles="--", zorder=3)
        ax.contour(r["gx"], r["gy"], r["d5"],
                   levels=[hdr_level(r["d5"], r["cell_area"])],
                   colors=GEN5_COLOR, linewidths=1.8, zorder=3)

        _style(ax, xk, yk, f"{xk} / {yk}", r["extent"])
        ax.title.set_fontsize(13)
        ax.xaxis.label.set_fontsize(11)
        ax.yaxis.label.set_fontsize(11)

        # attach the bar to the axes' *drawn* area — with aspect="equal" the
        # subplot box is wider than the image, and a normal colorbar would
        # float off to the right of it
        cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.12)
        cb = fig.colorbar(im, cax=cax)
        cb.ax.tick_params(labelsize=8, colors=MUTED, length=2)
        cb.outline.set_visible(False)
        cb.set_label("Δ density", fontsize=10, color=MUTED)

    fig.legend(handles=[
        Line2D([], [], color=GEN1_COLOR, lw=1.6, ls="--",
               label=f"{a_label} — {BOUND_MASS:.0%} of vases"),
        Line2D([], [], color=GEN5_COLOR, lw=1.8,
               label=f"{b_label} — {BOUND_MASS:.0%} of vases")],
        loc="lower center", ncol=2, frameon=False, fontsize=10,
        labelcolor=INK, bbox_to_anchor=(0.5, 0.005))

    # fig.suptitle("Density Difference",
    #              fontsize=15, color=INK, y=0.99)
    fig.tight_layout(rect=(0, 0.06, 1, 0.9))
    fig.savefig(FIG_DIR / filename, dpi=200, facecolor=SURFACE)
    plt.show()


plot_diff_planes(results, "Real (human choice)", "pc_density_diff_planes.png",
                 a_label="Null run (random choice)")

### The same map for the final choice ("king of kings") vases

Same switch as above, applied to the final choice: the baseline is now the **null run's
own final choices** rather than gen 1. Each null repeat ran the same "pick the most
beautiful of your five" step over its own randomly-grown vases, so subtracting it
cancels the generative drift and leaves the effect of choosing.

Two things differ from the gen-5 comparison:

- **The block is the participant, not the lineage.** For gen-5 selections both runs have
  a vase in every round, so each real vase can be paired with the null vases grown from
  the same gen-1 shapes. Here, *which round wins is itself part of what is being decided*
  — the real participant might crown their round-1 vase while a null repeat crowns
  round 4 — so there is no round to match on and the participant is the exchangeable
  unit: 1 real choice against 10 null choices.
- **n is 2,724 real against 27,240 null.** One vase per participant per run, so this is
  the only comparison here with no repeated measures on the real side.

The round-preference readout printed above is worth a look on its own: the null crowns
each round about 20% of the time, as it must, while real participants favour round 1.
That is a primacy effect in the human data that no amount of density mapping would have
surfaced.


In [ ]:
# Same switch as above: baseline is the null run's own final choices, so the
# generative drift cancels and what is left is the effect of choosing.
results_final = compute_planes(null_final_choice, final_choice)
report_planes(results_final, "real")

plot_diff_planes(results_final, "Real final choice",
                 "pc_density_diff_planes_final_choice.png",
                 a_label="Null final choice (random)")

## Permutation test

The maps above are descriptive — they show a difference but say nothing about whether it
could have arisen by chance. This tests it.

**Null hypothesis.** Within a participant, their five real gen-5 selections and their
five null-run gen-5 selections are exchangeable — which vases came from which run is
arbitrary, and the density difference is noise.

**Why labels are shuffled *within* participant.** A plain vase-level shuffle would treat
all 27,240 vases as independent draws. They are not: each participant contributes five
vases to each run, and the two runs share the same people and the same gen-1 starting
vases. Shuffling across participants would break that pairing, make the null distribution
far too narrow, and give an anti-conservative p-value. Permuting labels inside each
participant's own block respects the matched design and tests the question actually being
asked. This comparison is better balanced than the old gen-1 one: every participant
contributes exactly five vases to each side, so the permutation is a clean 5-vs-5 swap.

**Two statistics**, both from the same permuted surfaces:

- `max |Δdensity|` — the largest difference anywhere on the grid. Its null distribution
  gives a **simultaneous** threshold: any cell exceeding the 95th percentile of the null
  maxima is significant at a family-wise error rate of 5% across the whole 220×220 grid.
  This is the Westfall–Young max-statistic approach, and it is what makes a per-pixel
  significance map legitimate despite 48,400 simultaneous comparisons.
- **Integrated squared difference**, `∫(Δdensity)² dA` — one global number for how far
  apart the two distributions are overall.

p-values use `(1 + #{null ≥ observed}) / (1 + B)`, so the smallest attainable value is
`1/(B+1)`; a p at that floor means no permutation ever reached the observed statistic,
not that p is zero.


In [ ]:
# The figures use the binned KDE (KDE_MODE above). This checks it against the
# exact scipy estimator on the real comparison, so the speed-up is validated
# rather than assumed. Computing the exact surface here is the only place the
# slow path still runs — twice per plane, not thousands of times.
for xk, yk in PLANES:
    r = results[(xk, yk)]
    a = null_gen5_select[[xk, yk]].to_numpy()
    b = gen5_select[[xk, yk]].to_numpy()
    pooled = np.vstack([a, b])
    mu, sd = pooled.mean(0), pooled.std(0)
    exact = (kde_grid(b, mu, sd, r["bw"], r["gx"], r["gy"])
             - kde_grid(a, mu, sd, r["bw"], r["gx"], r["gy"]))
    corr = np.corrcoef(exact.ravel(), r["diff"].ravel())[0, 1]
    err = np.abs(r["diff"] - exact).max() / np.abs(exact).max()
    print(f"{xk}/{yk}: binned vs exact difference surface — "
          f"corr={corr:.5f}, max discrepancy={err:.1%} of peak")


In [ ]:
N_PERM = 1000    # permutations drawn; the smallest reachable p is 1/(N_PERM+1)
ALPHA = 0.05     # family-wise error rate the significance mask controls


def permute_within_block(codes_sorted, starts_rep, na_rep, rng):
    """Draw one permutation: reshuffle the A/B labels INSIDE each block.

    A block is one matched set — for the gen-5 test, a single lineage's
    1 real champion and its 10 null-run counterparts, all grown from the same
    generation-1 pair by the same generator. Under the null hypothesis those 11
    are interchangeable, so relabelling which of them is "real" produces a
    dataset that is just as likely as the one observed.

    Every argument is precomputed once by the caller and reused for all 1,000
    draws, because none of it depends on the random numbers.

    codes_sorted : (n,) block id per row, already sorted so each block's rows
                   are contiguous.
    starts_rep   : (n,) index of the first row of THIS row's block.
    na_rep       : (n,) how many rows of this block belong to group A (10).
    """
    n = codes_sorted.size

    # Sort by block first, by a fresh random key second. Because the primary key
    # is the block, rows never move between blocks — the shuffle happens only
    # WITHIN each block, which is exactly the exchangeability the null claims.
    order = np.lexsort((rng.random(n), codes_sorted))

    # `order` lists the rows in (block, random) order. Position i in that list
    # minus the start of its block gives the row's rank inside its own block,
    # 0..10 — a random rank, since the tie-break was random.
    rank_in_block = np.arange(n) - starts_rep

    # Take the first na_rep ranks of each block as group A and the rest as B.
    # Writing through `order` scatters those labels back to the original rows.
    # This preserves each block's A/B split exactly (10 null, 1 real), so every
    # permuted dataset has the same group sizes as the observed one — which
    # matters, because both test statistics depend on sample size.
    lab_a = np.empty(n, dtype=bool)
    lab_a[order] = rank_in_block < na_rep
    return lab_a


def permutation_test(a_df, b_df, res, n_perm=N_PERM, alpha=ALPHA, seed=0,
                     block_col="block"):
    """Is the B landscape different from the A landscape, beyond chance?

    a_df is the baseline (null run), b_df the comparison (real run). Runs one
    independent test per plane and returns, for each: two global p-values and a
    boolean mask of the cells that differ significantly.
    """
    rng = np.random.default_rng(seed)   # seeded, so the run is reproducible
    out = {}

    for xk, yk in res:                  # each PC plane is tested separately
        r = res[(xk, yk)]

        # ---- the observed difference surface --------------------------------
        a = a_df[[xk, yk]].to_numpy()   # baseline points, (n_a, 2)
        b = b_df[[xk, yk]].to_numpy()   # comparison points, (n_b, 2)
        pooled = np.vstack([a, b])      # both groups; fixes the common scale

        # Grid, bin edges and kernel width are derived ONCE from the pooled data
        # and then held fixed for the observed surface and all 1,000 permuted
        # ones. If they were recomputed per permutation the null distribution
        # would absorb variation in the grid rather than in the labels.
        ex, ey, sig_px, cell = grid_spec(r, r["bw"], pooled.std(0))

        # Each KDE integrates to 1, so the difference is a redistribution of
        # probability mass and the unequal group sizes (1 real : 10 null) do not
        # bias it. Sign convention: positive = more real than null.
        obs = (binned_kde(b, ex, ey, sig_px, cell)
               - binned_kde(a, ex, ey, sig_px, cell))

        # Two summaries of "how different", because they catch different shapes
        # of effect: a single sharp spike vs a broad shallow shift.
        obs_max = np.abs(obs).max()        # largest difference anywhere
        obs_isd = (obs ** 2).sum() * cell  # integrated squared difference

        # ---- set up the blocking (done once, reused by every permutation) ----
        # Stack both groups' block labels in the same row order as `pooled`.
        pid = np.concatenate([a_df[block_col].to_numpy(),
                              b_df[block_col].to_numpy()])

        # Map the label strings to small integers so they can be counted and
        # sorted cheaply.
        codes = pd.factorize(pid)[0]

        # Sort rows so each block is contiguous. `kind="stable"` keeps the
        # within-block order deterministic, so the only randomness later comes
        # from the generator.
        order0 = np.argsort(codes, kind="stable")
        codes_s, xy_s = codes[order0], pooled[order0]

        # Which sorted rows are baseline (True) vs comparison (False).
        is_a_s = np.concatenate([np.ones(len(a), bool),
                                 np.zeros(len(b), bool)])[order0]

        counts = np.bincount(codes_s)                        # rows per block (11)
        starts = np.concatenate([[0], np.cumsum(counts)[:-1]])  # each block's first row
        starts_rep = np.repeat(starts, counts)               # broadcast to every row
        # How many baseline rows each block holds (10) — broadcast the same way,
        # so the permutation can preserve the split block by block.
        na_rep = np.repeat(np.bincount(codes_s, weights=is_a_s).astype(int), counts)

        # ---- build the null distribution ------------------------------------
        null_max = np.empty(n_perm)
        null_isd = np.empty(n_perm)
        for i in range(n_perm):
            # Relabel which vases are "real" within each block...
            lab_a = permute_within_block(codes_s, starts_rep, na_rep, rng)
            # ...and rebuild the difference surface from the relabelled groups,
            # using the SAME grid and kernel as the observed one.
            dperm = (binned_kde(xy_s[~lab_a], ex, ey, sig_px, cell)
                     - binned_kde(xy_s[lab_a], ex, ey, sig_px, cell))
            null_max[i] = np.abs(dperm).max()
            null_isd[i] = (dperm ** 2).sum() * cell

        # ---- threshold and p-values -----------------------------------------
        # The 95th percentile of the null MAXIMA. Under the null, the largest
        # difference anywhere exceeds this 5% of the time — so the chance that
        # ANY of the 48,400 cells exceeds it is also 5%. One threshold controls
        # the whole grid simultaneously (Westfall-Young); no per-cell correction
        # is needed and no per-cell p-values are produced.
        thresh = np.quantile(null_max, 1 - alpha)

        out[(xk, yk)] = dict(
            obs_max=obs_max, obs_isd=obs_isd,
            null_max=null_max, null_isd=null_isd, thresh=thresh,
            # The +1 counts the observed labelling as one of its own
            # permutations, which is what makes the test exact rather than
            # optimistic. It also floors p at 1/(n_perm+1): a p of 0.001 means
            # "no permutation ever matched the observation", not "p = 0".
            p_max=(1 + (null_max >= obs_max).sum()) / (1 + n_perm),
            p_isd=(1 + (null_isd >= obs_isd).sum()) / (1 + n_perm),
            # Cells whose observed difference clears the simultaneous threshold.
            # r["diff"] is built by compute_planes from the same grid, bandwidth
            # and kernel as `obs` above, so the mask lines up exactly with the
            # surface the figures draw.
            sig=np.abs(r["diff"]) > thresh,
        )
    return out


def report_perm(perm, label):
    print(f"\n{label}  ({N_PERM:,} permutations, FWER {ALPHA:.0%})")
    for (xk, yk), p in perm.items():
        print(f"  {xk}/{yk}: max|Δ|={p['obs_max']:.5f} vs null 95th pct "
              f"{p['thresh']:.5f} (null max ever {p['null_max'].max():.5f})  "
              f"p_max={p['p_max']:.4f}  p_isd={p['p_isd']:.4f}  "
              f"significant area={p['sig'].mean():.1%}")


perm_gen5 = permutation_test(null_gen5_select, gen5_select, results, seed=0)
report_perm(perm_gen5, "Real vs null gen-5 selections")

perm_final = permutation_test(null_final_choice, final_choice, results_final, seed=1)
report_perm(perm_final, "Real vs null final choices")


### One p-value per comparison

The test above gives a p-value per plane. This collapses them into a single omnibus
p-value answering: **does human selection move the landscape away from what random
selection produces, taken over PC1/PC2, PC3/PC4 and PC5/PC6 together?**

Every plane is scored on the *same* permutation draw. That is the important detail — the
planes are built from the same vases and the same participants, so their statistics are
correlated. Reusing the draw lets the null distribution absorb that correlation.
Combining separately-computed per-plane p-values instead (Fisher, Bonferroni, Šidák)
would assume independence that does not hold here, and would give the wrong answer.

Each plane's integrated squared difference is z-scored against its own permutation null
before they are summed, so the plane with the larger raw density scale cannot dominate
the combined statistic by accident.

With the null runs as the baseline, a significant result means something much stronger
than it did against gen 1: the generative process is held fixed on both sides, so a
difference is attributable to **choice**, not to the generator expanding.


In [ ]:
def omnibus_permutation(a_df, b_df, res, n_perm=N_PERM, seed=0,
                        block_col="block"):
    """ONE p-value per comparison, combining every plane in `res`.

    Every plane is scored on the *same* permutation draw. That matters: the
    planes share vases and participants, so their statistics are correlated.
    Reusing the draw lets the null distribution carry that correlation instead
    of assuming independence (which combining two separate p-values, e.g. by
    Fisher or Bonferroni, would wrongly do).

    Per-plane statistic is the integrated squared difference. The planes
    are on different density scales, so each is z-scored against its own null
    before summing — otherwise whichever plane has the larger raw ISD would
    silently dominate the combined test.
    """
    rng = np.random.default_rng(seed)
    planes = list(res)

    # blocking is identical for every plane (same rows)
    pid = np.concatenate([a_df[block_col].to_numpy(),
                          b_df[block_col].to_numpy()])
    codes = pd.factorize(pid)[0]
    order0 = np.argsort(codes, kind="stable")
    codes_s = codes[order0]
    is_a_s = np.concatenate([np.ones(len(a_df), bool),
                             np.zeros(len(b_df), bool)])[order0]
    counts = np.bincount(codes_s)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    starts_rep = np.repeat(starts, counts)
    na_rep = np.repeat(np.bincount(codes_s, weights=is_a_s).astype(int), counts)

    specs, obs_isd = [], []
    for xk, yk in planes:
        r = res[(xk, yk)]
        a = a_df[[xk, yk]].to_numpy()
        b = b_df[[xk, yk]].to_numpy()
        pooled = np.vstack([a, b])
        ex, ey, sig_px, cell = grid_spec(r, r["bw"], pooled.std(0))
        specs.append((pooled[order0], ex, ey, sig_px, cell))
        obs = (binned_kde(b, ex, ey, sig_px, cell)
               - binned_kde(a, ex, ey, sig_px, cell))
        obs_isd.append((obs ** 2).sum() * cell)
    obs_isd = np.array(obs_isd)

    null = np.empty((n_perm, len(planes)))
    for i in range(n_perm):
        lab_a = permute_within_block(codes_s, starts_rep, na_rep, rng)
        for j, (xy_s, ex, ey, sig_px, cell) in enumerate(specs):
            dperm = (binned_kde(xy_s[~lab_a], ex, ey, sig_px, cell)
                     - binned_kde(xy_s[lab_a], ex, ey, sig_px, cell))
            null[i, j] = (dperm ** 2).sum() * cell

    # z-score each plane against its own null, then sum across planes
    mu, sd = null.mean(0), null.std(0)
    null_z = ((null - mu) / sd).sum(1)
    obs_z = ((obs_isd - mu) / sd).sum()

    return dict(
        planes=planes, obs_isd=obs_isd, obs_z=obs_z,
        per_plane_z=(obs_isd - mu) / sd, null_z=null_z,
        p=(1 + (null_z >= obs_z).sum()) / (1 + n_perm), n_perm=n_perm,
    )


def report_omnibus(om, label):
    planes = " + ".join(f"{x}/{y}" for x, y in om["planes"])
    print(f"{label}")
    print(f"  combined over {planes}")
    for (xk, yk), z in zip(om["planes"], om["per_plane_z"]):
        print(f"    {xk}/{yk}: ISD is {z:,.1f} SD above its permutation null")
    print(f"  combined statistic {om['obs_z']:,.1f} vs null max "
          f"{om['null_z'].max():.2f}  ->  p = {om['p']:.4f} "
          f"(floor 1/{om['n_perm'] + 1} = {1 / (om['n_perm'] + 1):.4f})\n")


omni_gen5 = omnibus_permutation(null_gen5_select, gen5_select, results, seed=10)
report_omnibus(omni_gen5, "Real vs null gen-5 selections")

omni_final = omnibus_permutation(null_final_choice, final_choice, results_final, seed=11)
report_omnibus(omni_final, "Real vs null final choices")

### Calibration: does the method invent signal when there is none?

Ten repeats buy a check the single null run could not provide. Hold out one repeat,
treat it as if it were the "real" data, and use the other nine as its baseline. Both
sides are now random, so **there is no effect to find**. If the pipeline is honest the
difference maps should be featureless and the omnibus p-values should scatter roughly
uniformly over (0, 1). If instead they come back at the 0.001 floor with structured red
and blue regions, the method is manufacturing signal and nothing else in this notebook
can be trusted.

This is the strongest available test of the machinery itself — the KDE, the shared
bandwidth, the blocking, the permutation scheme — because it is the one case where the
correct answer is known in advance.


In [ ]:
# Leave-one-out: repeat k as "signal", the other nine as baseline. Both random,
# so a well-behaved test should mostly FAIL to reject. Fewer permutations here
# since we only need the p-value, not a per-pixel map.
LOO_PERM = 200

loo_p = {}
for k in REPEATS:
    held = null_gen5_select[null_gen5_select["null_repeat"] == k]
    rest = null_gen5_select[null_gen5_select["null_repeat"] != k]
    # mode="grid": omnibus_permutation builds its own binned surfaces, so
    # computing densities here would be thrown away 10 times over
    res_k = compute_planes(rest, held, mode="grid")
    om_k = omnibus_permutation(rest, held, res_k, n_perm=LOO_PERM, seed=100 + k)
    loo_p[k] = om_k["p"]
    print(f"  repeat {k:2d} vs other nine: p = {om_k['p']:.3f}   "
          f"(per-plane z: " + ", ".join(f"{z:+.1f}" for z in om_k["per_plane_z"]) + ")")

_p = np.array(list(loo_p.values()))
print()
print(f"null-vs-null p-values: min={_p.min():.3f} median={np.median(_p):.3f} "
      f"max={_p.max():.3f}")
print(f"significant at 5%: {(_p < 0.05).sum()} of {len(_p)} "
      f"(expect about {0.05 * len(_p):.1f} by chance)")
print()
print("Compare with the real-vs-null result above: if these sit near the p floor and"
      "\nshow large z-scores, the difference the real run shows is not evidence of"
      "\nselection but an artefact of the method.")


In [ ]:
def plot_significance(res, perm, b_label, filename, a_label="Gen 1 baseline"):
    """Difference map with non-significant regions washed out."""
    fig, axes = plt.subplots(1, len(res), figsize=(8.0 * len(res), 8.6),
                             facecolor=SURFACE)

    for ax, (xk, yk) in zip(np.atleast_1d(axes), res):
        r, p = res[(xk, yk)], perm[(xk, yk)]
        dmax = np.abs(r["diff"]).max()

        ax.imshow(r["diff"], origin="lower", extent=r["extent"], cmap=DIVERGING,
                  vmin=-dmax, vmax=dmax, aspect="equal", zorder=0)
        # veil everything that did not clear the simultaneous threshold
        veil = np.zeros(r["diff"].shape + (4,))
        veil[..., :3] = 0.98
        veil[..., 3] = np.where(p["sig"], 0.0, 0.82)
        ax.imshow(veil, origin="lower", extent=r["extent"], aspect="equal",
                  zorder=1, interpolation="nearest")
        ax.contour(r["gx"], r["gy"], p["sig"].astype(float), levels=[0.5],
                   colors=INK, linewidths=1.0, zorder=2)

        _style(ax, xk, yk, f"{xk} / {yk}   ({p['sig'].mean():.0%} significant)",
               r["extent"])
        ax.title.set_fontsize(13)
        ax.xaxis.label.set_fontsize(11)
        ax.yaxis.label.set_fontsize(11)

    fig.text(0.5, 0.035,
             f"outlined = significant at FWER {ALPHA:.0%} "
             f"({N_PERM:,} within-participant permutations); "
             f"washed-out areas did not clear the threshold",
             ha="center", va="bottom", fontsize=10, color=INK)

    fig.suptitle(f"{b_label} − {a_label}: where the difference survives "
                 f"the permutation test",
                 fontsize=15, color=INK, y=0.99)
    fig.tight_layout(rect=(0, 0.075, 1, 0.9))
    fig.savefig(FIG_DIR / filename, dpi=200, facecolor=SURFACE)
    plt.show()


plot_significance(results, perm_gen5, "Real (human choice)",
                  "pc_density_diff_significance.png",
                  a_label="Null run (random choice)")
plot_significance(results_final, perm_final, "Real final choice",
                  "pc_density_diff_significance_final_choice.png",
                  a_label="Null final choice (random)")

## What the shapes actually look like

The maps say *where* in PC space human choice moves the population relative to random
choice; they don't say what those coordinates look like as pots. This overlays
reconstructed vase silhouettes on a grid across each plane, each one filled with the
Δdensity at its own position — so red vases are shapes people chose more often than
chance, blue vases less often.

**How each vase is built.** The PCA was fitted to 250 outline points per vase, stored as
one 500-long vector (250 x-coordinates then 250 y-coordinates). Any point in PC space
inverts straight back to an outline: `coords = mean + scores @ components`. In every
panel only that panel's own two PCs are set to the cell's position and **all four other
PCs are held at 0** — so the PC1/PC2 panel holds PC3–PC6 at 0, the PC3/PC4 panel holds
PC1, PC2, PC5, PC6 at 0, and the PC5/PC6 panel holds PC1–PC4 at 0. Each panel therefore
shows how shape varies along its own two axes with everything else at the mean, rather
than mixing in variation the panel isn't plotting.

**Tuning the grid.** `N_VASE_COLS` is the knob. The row count defaults to whatever keeps
the cells roughly square for that panel's aspect ratio; pass `n_rows=` to override.

**On aspect ratio.** The axes are `aspect="equal"`, so one data unit is the same length
on both axes. Every silhouette is therefore scaled by a *single* scalar — never
separately in x and y — which keeps each vase undistorted. That scalar is also shared
across every vase in a panel, so the sizes stay comparable: a vase drawn twice as tall as
its neighbour really is twice as tall. The scale is set by the largest vase in the panel,
so nothing overflows its cell.

These are reconstructions from a 6-component model, not photographs of the vases people
saw. They capture the shape variance the PCA retained and nothing beyond it.


In [ ]:
from matplotlib.colors import Normalize
from matplotlib.patches import Polygon
from matplotlib.collections import LineCollection

PCA_COMPONENTS_CSV = "../file_processing/pca_csv/pc_vals.csv"
PCA_MEANS_CSV = "../file_processing/pca_csv/pc_val_means.csv"

# Rows are PC1..PC6, columns are the 500 outline coordinates (250 x then 250 y)
PCA_COMPONENTS = pd.read_csv(PCA_COMPONENTS_CSV, index_col=0).to_numpy()
PCA_MEAN = pd.read_csv(PCA_MEANS_CSV)["pc_mean"].to_numpy()
PC_NAMES = [f"PC{i + 1}" for i in range(PCA_COMPONENTS.shape[0])]
assert PCA_COMPONENTS.shape[1] == PCA_MEAN.size, "components/means mismatch"


def vase_outline(**pc_values):
    """Invert the PCA back to a closed outline.

    Only the PCs named are set; every other PC stays at 0, i.e. at the mean
    shape. Returns (x, y) in the PCA's own shape units.
    """
    scores = np.zeros(len(PC_NAMES))
    for name, value in pc_values.items():
        scores[PC_NAMES.index(name)] = value
    coords = PCA_MEAN + scores @ PCA_COMPONENTS
    n = coords.size // 2
    return coords[:n], coords[n:]


N_VASE_COLS = 30

def plot_vase_grid(res, b_label, filename, n_cols=N_VASE_COLS, n_rows=None,
                   fill=0.86, a_label="Gen 1 baseline",
                   edge_sensitivity=False, sens_vmax=None):
    """Vase silhouettes on the difference map.

    Fill  = Δdensity at the vase's position (blue<->red).
    Edge  = CNN outward-normal sensitivity, when `edge_sensitivity` is set
            (purple<->green, a deliberately different ramp so the two
            encodings cannot be confused). See the caveat in the markdown:
            that is the MODEL's gradient, not verified human preference.
    """
    fig, axes = plt.subplots(1, len(res), figsize=(8.0 * len(res), 8.6),
                             facecolor=SURFACE)

    for ax, (xk, yk) in zip(np.atleast_1d(axes), res):
        r = res[(xk, yk)]
        x0, x1, y0, y1 = r["extent"]
        # default row count keeps the grid cells roughly square
        nr = n_rows or max(2, int(round(n_cols * (y1 - y0) / (x1 - x0))))
        cw, ch = (x1 - x0) / n_cols, (y1 - y0) / nr
        cx = x0 + (np.arange(n_cols) + 0.5) * cw
        cy = y0 + (np.arange(nr) + 0.5) * ch

        dmax = np.abs(r["diff"]).max()
        norm = Normalize(-dmax, dmax)

        # build every outline first, so one shared scale can be chosen.
        # raw_outlines keeps the UNCENTRED curve: the CNN normalises with a
        # per-channel shift, so translating a vase would change its score.
        shapes, values, raw_outlines = [], [], []
        for yy in cy:
            for xx in cx:
                vx, vy = vase_outline(**{xk: xx, yk: yy})
                raw_outlines.append(np.stack([vx, vy]))
                shapes.append((xx, yy, vx - vx.mean(), vy - vy.mean()))
                # nearest cell of the KDE grid (regular, so index directly)
                ix = int(round((xx - x0) / (x1 - x0) * (GRID_N - 1)))
                iy = int(round((yy - y0) / (y1 - y0) * (GRID_N - 1)))
                values.append(r["diff"][iy, ix])

        # ONE scalar for both axes and for every vase: no distortion, and
        # relative sizes stay honest between cells
        half_w = max(np.abs(s[2]).max() for s in shapes)
        half_h = max(np.abs(s[3]).max() for s in shapes)
        scale = min(cw * fill / (2 * half_w), ch * fill / (2 * half_h))

        ax.imshow(r["diff"], origin="lower", extent=r["extent"], cmap=DIVERGING,
                  vmin=-dmax, vmax=dmax, aspect="equal", zorder=0, alpha=0.18)
        # in the sensitivity variant the HDR contours go neutral grey, so the
        # green of the comparison contour cannot be read as an edge value
        c1 = MUTED if edge_sensitivity else GEN1_COLOR
        c5 = MUTED if edge_sensitivity else GEN5_COLOR
        ax.contour(r["gx"], r["gy"], r["d1"],
                   levels=[hdr_level(r["d1"], r["cell_area"])],
                   colors=c1, linewidths=1.2, linestyles="--",
                   alpha=0.5, zorder=1)
        ax.contour(r["gx"], r["gy"], r["d5"],
                   levels=[hdr_level(r["d5"], r["cell_area"])],
                   colors=c5, linewidths=1.3, alpha=0.6, zorder=1)

        sens = (outline_sensitivity(np.stack(raw_outlines).astype(np.float32))
                if edge_sensitivity else None)
        if edge_sensitivity:
            svmax = sens_vmax or float(np.percentile(np.abs(sens), 98))
            snorm = Normalize(-svmax, svmax)

        for j, ((xx, yy, vx, vy), value) in enumerate(zip(shapes, values)):
            pts = np.column_stack([xx + vx * scale, yy + vy * scale])
            ax.add_patch(Polygon(pts, closed=True, facecolor=DIVERGING(norm(value)),
                                 edgecolor="none" if edge_sensitivity else INK,
                                 linewidth=0.5, zorder=3))
            if edge_sensitivity:
                loop = np.vstack([pts, pts[:1]])
                segs = np.stack([loop[:-1], loop[1:]], axis=1)
                v = np.append(sens[j], sens[j][0])
                lc = LineCollection(segs, cmap=SENS_CMAP, norm=snorm,
                                    linewidths=1.3, capstyle="round", zorder=4)
                lc.set_array(0.5 * (v[:-1] + v[1:]))
                ax.add_collection(lc)

        _style(ax, xk, yk, f"{xk} / {yk}   ({n_cols}x{nr} vases)", r["extent"])
        ax.title.set_fontsize(13)
        ax.xaxis.label.set_fontsize(11)
        ax.yaxis.label.set_fontsize(11)

        cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.12)
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=DIVERGING), cax=cax)
        cb.ax.tick_params(labelsize=8, colors=MUTED, length=2)
        cb.outline.set_visible(False)
        cb.set_label("Δ density at the vase's position", fontsize=10, color=MUTED)

    if edge_sensitivity:
        cax2 = fig.add_axes([0.35, 0.055, 0.30, 0.015])
        cb2 = fig.colorbar(plt.cm.ScalarMappable(norm=snorm, cmap=SENS_CMAP),
                           cax=cax2, orientation="horizontal")
        cb2.ax.tick_params(labelsize=7, colors=MUTED, length=2)
        cb2.outline.set_visible(False)
        cb2.set_label("outline: CNN d(logit)/d(outward displacement)  "
                      "— green = bulging out raises the model's score",
                      fontsize=9, color=MUTED)

    fig.legend(handles=[
        Line2D([], [], color=MUTED if edge_sensitivity else GEN1_COLOR,
               lw=1.2, ls="--",
               label=f"{a_label} — {BOUND_MASS:.0%} of vases"),
        Line2D([], [], color=MUTED if edge_sensitivity else GEN5_COLOR, lw=1.3,
               label=f"{b_label} — {BOUND_MASS:.0%} of vases")],
        loc="lower center", ncol=2, frameon=False, fontsize=10,
        labelcolor=INK, bbox_to_anchor=(0.5, 0.005))

    fig.suptitle(f"Vase shapes across PC space, filled by {b_label} − {a_label} "
                 f"density difference\n(all other PCs held at 0)",
                 fontsize=15, color=INK, y=0.99)
    fig.tight_layout(rect=(0, 0.11 if edge_sensitivity else 0.06, 1, 0.91))
    fig.savefig(FIG_DIR / filename, dpi=200, facecolor=SURFACE)
    plt.show()


plot_vase_grid(results, "Real (human choice)", "pc_vase_grid.png",
               a_label="Null run (random choice)")
plot_vase_grid(results_final, "Real final choice",
               "pc_vase_grid_final_choice.png",
               a_label="Null final choice (random)")

## Overlaying the CNN's shape attribution on the outline

The fill of each vase says where in PC space human choice concentrates the population.
That is a statement about *positions*. It says nothing about **which part of a given
silhouette** is doing the work. The 1D CNN in `neural_net/vase_select_cnn.ipynb` answers
that second question, so the two can be drawn together: **fill = Δdensity, outline = the
CNN's outward-normal sensitivity**, on a deliberately different colour ramp
(purple↔green) so the two encodings cannot be confused.

**What the outline colour is.** For each point on the silhouette,
`d(logit) / d(outward displacement)` — if this point bulged outward, would the model
score the vase as more likely to be chosen? Green = yes, purple = no, neutral grey =
this point does not move the score. It is the same `nsens` quantity the CNN notebook
plots in its Step 11 figures, computed here on the PCA-reconstructed grid vases.

**Three things make this join sound**, and each is checked rather than assumed:

1. **The PCA reconstruction is exact.** Reconstructing a real game vase from its six PCs
   reproduces its stored outline to RMSE 0.0 — same coordinate space, same 250-point
   traversal order the CNN was trained on. No rescaling or resampling is needed.
2. **The checkpoint is the half-contour model.** `vase_select_cnn.pt` reproduces its own
   saved test logits to 0.000000 with `HALF = slice(1, 126)` and `padding_mode="reflect"`.
   (Its `arch` string says "circular padding" — that label is stale, the weights are not.)
3. **Sensitivity is computed on the uncentred outline.** The CNN normalises with a
   per-channel shift, so translating a vase changes its score; the grid centres each
   silhouette only for drawing. The mirror-symmetry check confirms it: sensitivity at
   index `i` equals sensitivity at its mirror index to 0.0.

### The caveat that matters most

**Do not read the outline colour as "this part of the vase is selected for."** The CNN
notebook tests exactly that claim in its Step 12 and finds it does not hold: against a
logistic regression fitted directly to the choices, the CNN's sensitivity map correlates
at **r ≈ +0.07** across height bands with sign agreement of **60%** against a 50% chance
baseline. The model itself is weak in absolute terms — ROC-AUC 0.602, pairwise accuracy
0.648 against a 0.595 parent-repetition baseline.

So the honest reading is: *this is where the model's gradient points*, which is a
description of the model, not a validated description of what people prefer. The fill —
built from 136,200 pooled null vases and a calibrated permutation test — is on much
firmer ground than the outline. They are shown together because the question is natural,
not because they carry equal evidential weight.

One genuine cross-check does survive: the CNN scores the tall narrow vase at PC1 = −10
higher (logit +0.58) than the wide bowl at PC1 = +10 (−0.22), which agrees in direction
with the density difference from a completely independent method.

**Grid size.** Outline colour needs pixels to read, so this figure defaults to
`N_VASE_COLS_CAM` (12) rather than the 30 used for the fill-only grid.


In [ ]:
# --- the CNN from neural_net/vase_select_cnn.ipynb -------------------------
import torch
import torch.nn as nn

CNN_CKPT = "../neural_net/vase_select_cnn.pt"

FULL_LEN = 250
MIRROR = (1 - np.arange(FULL_LEN)) % FULL_LEN
HALF = slice(1, 126)   # one representative of each mirror pair, rim -> foot
_ar = np.arange(FULL_LEN)
HALF_TO_FULL = np.where((_ar >= 1) & (_ar <= 125), _ar, MIRROR) - 1


class VaseCNN(nn.Module):
    """Must match neural_net/vase_select_cnn.ipynb exactly to load its weights."""

    def __init__(self, in_channels=2, dropout=0.1, pad_mode="reflect"):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, 7, padding=3, padding_mode=pad_mode),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2, padding_mode=pad_mode),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1, padding_mode=pad_mode),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 256, 3, padding=1, padding_mode=pad_mode),
            nn.BatchNorm1d(256), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(256, 128), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        return self.head(self.conv(x)).squeeze(1)


_ck = torch.load(CNN_CKPT, map_location="cpu", weights_only=False)
cnn = VaseCNN(); cnn.load_state_dict(_ck["state_dict"]); cnn.eval()
CNN_MU = np.asarray(_ck["norm_stats"]["mu"], dtype=np.float32)
CNN_SCALE = np.float32(_ck["norm_stats"]["scale"])
print("CNN loaded — test " + ", ".join(
    f"{k} {v:.3f}" for k, v in _ck["test_metrics"].items() if isinstance(v, float)))
print(f"parent-repetition baseline: {_ck['parent_baseline_test']:.3f} "
      "(what pairwise accuracy must beat to mean anything)")


def outward_normals(xy_full):
    """Unit outward normal per point of a closed contour (2, 250).

    Orientation from the polygon's signed area, not a centroid test — the
    centroid test is wrong wherever the outline is non-convex, i.e. at exactly
    the necks and waists where the signal lives.
    """
    P = xy_full.T
    T = np.roll(P, -1, axis=0) - np.roll(P, 1, axis=0)
    T /= np.linalg.norm(T, axis=1, keepdims=True) + 1e-9
    N = np.stack([T[:, 1], -T[:, 0]], axis=1)
    area = 0.5 * np.sum(P[:, 0] * np.roll(P[:, 1], -1)
                        - np.roll(P[:, 0], -1) * P[:, 1])
    return N * np.sign(area)


def outline_sensitivity(contours_full, batch=512):
    """(n, 2, 250) raw-unit outlines -> (n, 250) d(logit)/d(outward displacement).

    Positive = bulging outward here would RAISE the model's score. Computed on
    the half-profile the model actually sees, then mirrored back onto all 250
    points, which is exact because the two halves are the same curve.
    """
    Xh = np.ascontiguousarray(contours_full[:, :, HALF]).astype(np.float32)
    Xn = (Xh - CNN_MU.reshape(1, 2, 1)) / (CNN_SCALE + 1e-8)
    grads = np.empty_like(Xn)
    for s in range(0, len(Xn), batch):
        xb = torch.from_numpy(Xn[s:s + batch]).requires_grad_(True)
        g, = torch.autograd.grad(cnn(xb).sum(), xb)
        grads[s:s + batch] = g.numpy()
    out = np.empty((len(Xn), Xh.shape[2]), dtype=np.float32)
    for i in range(len(Xn)):
        out[i] = np.einsum("ci,ic->i", grads[i], outward_normals(contours_full[i])[HALF])
    return out[:, HALF_TO_FULL]


def cnn_logit(contours_full):
    Xh = np.ascontiguousarray(contours_full[:, :, HALF]).astype(np.float32)
    Xn = (Xh - CNN_MU.reshape(1, 2, 1)) / (CNN_SCALE + 1e-8)
    with torch.no_grad():
        return cnn(torch.from_numpy(Xn)).numpy()


# --- checks, not assumptions ----------------------------------------------
_probe = np.stack([np.stack(vase_outline(**kw)) for kw in
                   ({}, dict(PC1=-10), dict(PC1=10))]).astype(np.float32)
_s = outline_sensitivity(_probe)
assert np.abs(_s - _s[:, MIRROR]).max() < 1e-6, "sensitivity is not mirror-symmetric"
print(f"mirror-symmetry of sensitivity: max deviation "
      f"{np.abs(_s - _s[:, MIRROR]).max():.1e} (exact)")
_lg = cnn_logit(_probe)
print(f"CNN logit — mean vase {_lg[0]:+.3f}, PC1=-10 (tall narrow) {_lg[1]:+.3f}, "
      f"PC1=+10 (wide bowl) {_lg[2]:+.3f}")

# Purple<->green: a second diverging ramp with a neutral midpoint, chosen to be
# unmistakable against the blue<->red used for the density difference.
SENS_CMAP = LinearSegmentedColormap.from_list("purple_green", [
    "#40004b", "#762a83", "#9970ab", "#c2a5cf", "#e7d4e8",
    "#f0efec",
    "#d9f0d3", "#a6dba0", "#5aae61", "#1b7837", "#00441b"])


In [ ]:
# Outline colour needs pixels; 30 columns makes the vases too small to read one.
N_VASE_COLS_CAM = 12

plot_vase_grid(results, "Real (human choice)", "pc_vase_grid_sensitivity.png",
               a_label="Null run (random choice)",
               n_cols=N_VASE_COLS_CAM, edge_sensitivity=True)

plot_vase_grid(results_final, "Real final choice",
               "pc_vase_grid_sensitivity_final_choice.png",
               a_label="Null final choice (random)",
               n_cols=N_VASE_COLS_CAM, edge_sensitivity=True)


### Reading the map, and what it does not show

- **The generator confound is controlled, not just flagged.** Against a gen-1 baseline
  most of the signal was the generator widening the distribution over five generations,
  which would have happened with no preference at all. Both sides of these comparisons
  have been through the same five generations of the same generator from the same
  starting vases, so that expansion cancels. What survives is attributable to choosing.
- **The baseline is now ten pooled repeats, not one.** Pooling averages the ten null
  density surfaces while keeping every vase's true position, which cuts the baseline's
  sampling noise without distorting its spread. (Averaging the ten runs' *PC values*
  would have been the tempting mistake: it shrinks run-to-run variance tenfold and
  produces an artificially tight null cloud, which would have reversed the finding that
  random selection spreads wider than human selection.) The residual noise is now almost
  entirely in the real run, which is the floor this design can reach.
- **The permutation blocks on the lineage for gen-5, the participant for final choice.**
  Each round grows from its own gen-1 pair, so round-0 and round-3 vases are not
  interchangeable even under the null; blocking on (participant, round) pairs each real
  vase with the ten null vases grown from the same starting shapes. The earlier
  participant-level blocking was valid but conservative, since it let permuted groups
  become unbalanced across rounds.
- **Read the leave-one-out calibration before trusting any of this.** It is the only
  test here whose correct answer is known in advance.
- **Significance is not size.** The test has substantial power, so a difference can clear
  FWER 5% and still be small. The Δdensity colourbar, not the significance mask, says how
  big the effect is.
- **The planes carry very different amounts of variance** — roughly 52% of gen-1 shape
  variance in PC1/PC2, 35% in PC3/PC4 and only 13% in PC5/PC6 (printed above). Each map
  is scaled to its own range, so a saturated red in the PC5/PC6 panel is a much smaller
  change in the actual pot than the same red in PC1/PC2. Compare colourbars, not colours.
- **The null surfaces use the binned KDE**, which agrees with the exact estimator at
  r ≈ 0.999 but differs by a few percent of peak at the sharpest ridges. The significance
  boundary is therefore approximate to within about a pixel.


## Movement through the landscape

Everything above is a *static* comparison: where the two populations ended up, at gen 5 or at
the final choice. It says nothing about how anyone got there. The arrows below add that.

Each arrow is the **mean one-generation displacement of the champion** for every lineage whose
champion sits in that cell — champions are binned by where they *start*, and the arrow is the
average of `(next position − current position)` over those transitions. The grid uses the same
window as the density map underneath, so the two overlay exactly. Arrow direction is where
champions went next; arrow length is how far, in PC units per generation.

Two versions, because the arrows and the landscape have to be asking the same question:

- **Real drift** — where champions actually moved. Directly observed, no baseline.
- **Real − null drift** — the same field with the null run's drift subtracted, cell by cell.
  The landscape underneath is already `real − null`, so this is the version that matches it: it
  shows movement attributable to *choosing*, with the part the generator would have produced
  anyway removed. Where the two versions differ, the raw arrows were partly the generator.

Things to keep in mind when reading them:

- **Retentions are included.** When the incumbent wins it is re-recorded at the next generation
  with identical PCs, so that step is exactly zero — 62% of all steps. Dropping them would bias
  every arrow towards motion, so a short arrow genuinely means "champions here tend to stay".
- **It is a vector mean**, so opposing movements cancel. A short arrow can mean "they stay put"
  *or* "they scatter evenly in all directions"; the two are indistinguishable here.
- **Cells with fewer than 50 transitions are left blank**, and steps starting outside the window
  are dropped rather than clipped into edge cells. Sparse edge cells otherwise show spurious
  inward flow simply because all their populated neighbours lie inward.
- **Pooled arrow length is confounded with generation.** Step size collapses over a session
  (mean 1.83 → 1.13 → 0.71 → 0.44 PC units across the four steps, which is the convergence the
  game is designed to produce), and cells differ in which generation their champions tend to be
  at — on PC1/PC2, `corr(mean source generation, mean step length) = −0.79` across cells. So in
  the pooled figure a long arrow partly means "this region holds early-generation champions"
  rather than "this region pulls hard". Direction is far less affected than magnitude.

**Set `FACET_BY_GENERATION = True`** in the next cell to split the figure into one row per step
(1→2, 2→3, 3→4, 4→5). That removes the confound — every arrow in a row is the same step — and
shows the convergence directly: the field visibly collapses from the top row to the bottom. The
arrow magnification is deliberately **shared down each column** so that collapse is real rather
than renormalised away row by row.

Arrow appearance is controlled by the settings block at the top of the next cell: `ARROW_STYLE`
(colour, shaft width, head shape, alpha), `ARROW_N` (grid resolution), `ARROW_MIN_COUNT`, and
`ARROW_FILL` / `ARROW_GAIN` for the magnification. Arrows have to be magnified to be legible —
the drift is a few tenths of a PC unit against a plane 14–34 units wide — so each column carries
a scale key giving the true magnitude.

In [ ]:
# `play_counter` is round + 1, so (participant_id, play_counter) identifies one lineage;
# the round in `generation_key` is redundant with it (checked: 1:1 in both runs).
REAL_KEYS = ["participant_id", "play_counter"]
NULL_KEYS = ["null_repeat", "participant_id", "play_counter"]


def champion_steps(df, keys):
    """Champion at generation n -> the same lineage's champion at generation n+1.

    Retentions are kept. When the incumbent wins it reappears at the next generation under a
    new vase row with identical PC values, i.e. a step of exactly zero length. Those are
    genuine dynamics — the state was observed not to move — and dropping them would bias the
    drift field towards motion.
    """
    champs = (df[df["is_selected_vase"]]
              .pivot_table(index=keys + ["generation_key"], columns="pc_name",
                           values="pc_value")
              .reset_index())
    champs["gen"] = champs["generation_key"].str.extract(r"gen_(\d+)").astype(int)
    # align each champion with the one that follows it, within the same lineage
    nxt = champs.drop(columns="generation_key").assign(gen=lambda d: d["gen"] - 1)
    return champs.merge(nxt, on=keys + ["gen"], suffixes=("", "_next"))


real_steps = champion_steps(pc_clean, REAL_KEYS)
null_steps = champion_steps(null_data, NULL_KEYS)

assert len(real_steps) == 2724 * 5 * 4, "real lineages did not resolve to 4 steps per round"
assert len(null_steps) == len(REPEATS) * 2724 * 5 * 4, "null lineages did not resolve"

_moved = np.hypot(real_steps["PC1_next"] - real_steps["PC1"],
                  real_steps["PC2_next"] - real_steps["PC2"]) > 1e-12
print(f"real champion steps: {len(real_steps):,}  "
      f"({(~_moved).mean():.1%} retentions, i.e. zero-length)")
print(f"null champion steps: {len(null_steps):,}  ({len(REPEATS)} repeats pooled)")

In [ ]:
# ── Arrow settings — edit these ──────────────────────────────────────────────
FACET_BY_GENERATION = False
# False -> one row, all four steps pooled.
# True  -> one row per generation step (1->2, 2->3, 3->4, 4->5). The arrow scale is
#          shared down each column, so the steps visibly shrink from row to row
#          instead of each row being renormalised to look the same size.

ARROW_STYLE = dict(
    color="#0b0b0b",      # any matplotlib colour
    width=0.0042,         # shaft thickness, as a fraction of the panel width
    headwidth=3.4,        # head width, in multiples of the shaft width
    headlength=3.8,       # head length, same units
    headaxislength=3.2,   # where the head meets the shaft; < headlength gives a swept back barb
    alpha=1.0,
)

ARROW_N = 25              # cells along the longer axis; cells are kept ~square
ARROW_MIN_COUNT = 30      # a cell needs this many transitions before an arrow is drawn
ARROW_FILL = 1.2          # target length of the longest arrows, in cell widths
ARROW_GAIN = "auto"       # "auto" uses ARROW_FILL; or set a number to fix the magnification
# ─────────────────────────────────────────────────────────────────────────────


def drift_field(steps, xk, yk, extent, gen=None, n=ARROW_N, min_count=ARROW_MIN_COUNT):
    """Mean observed displacement per cell, on the density map's own window.

    `gen` selects a single source generation (1 means the 1->2 step); None pools all of
    them. Steps starting outside `extent` are dropped rather than clipped, because
    clipping would pile every outlier into the edge cells and point them artificially
    inward.
    """
    if gen is not None:
        steps = steps[steps["gen"] == gen]

    x0, x1, y0, y1 = extent
    span_x, span_y = x1 - x0, y1 - y0
    # square-ish cells, since the panels are drawn with aspect="equal"
    nx, ny = ((n, max(4, round(n * span_y / span_x))) if span_x >= span_y
              else (max(4, round(n * span_x / span_y)), n))

    px, py = steps[xk].to_numpy(), steps[yk].to_numpy()
    dx = steps[f"{xk}_next"].to_numpy() - px
    dy = steps[f"{yk}_next"].to_numpy() - py

    ex, ey = np.linspace(x0, x1, nx + 1), np.linspace(y0, y1, ny + 1)
    ix, iy = np.searchsorted(ex, px) - 1, np.searchsorted(ey, py) - 1
    inside = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny)
    ix, iy, dx, dy = ix[inside], iy[inside], dx[inside], dy[inside]

    U, V, C = (np.zeros((ny, nx)) for _ in range(3))
    np.add.at(U, (iy, ix), dx)
    np.add.at(V, (iy, ix), dy)
    np.add.at(C, (iy, ix), 1)

    enough = C >= min_count
    U = np.where(enough, U / np.maximum(C, 1), np.nan)
    V = np.where(enough, V / np.maximum(C, 1), np.nan)
    gx, gy = np.meshgrid(0.5 * (ex[:-1] + ex[1:]), 0.5 * (ey[:-1] + ey[1:]))
    return dict(gx=gx, gy=gy, U=U, V=V, count=C, n_outside=int((~inside).sum()))


def drift_rows(steps, facet=None):
    """[(row_label, {plane: field})] — one row pooled, or one per generation step."""
    facet = FACET_BY_GENERATION if facet is None else facet
    if not facet:
        return [(None, {p: drift_field(steps, *p, results[p]["extent"]) for p in PLANES})]
    return [(f"gen {g} → {g + 1}",
             {p: drift_field(steps, *p, results[p]["extent"], gen=g) for p in PLANES})
            for g in sorted(steps["gen"].unique())]


def subtract_rows(rows_a, rows_b):
    """Row-wise a − b, kept only where both runs have enough transitions in the cell."""
    return [(label, {p: dict(gx=a[p]["gx"], gy=a[p]["gy"],
                             U=a[p]["U"] - b[p]["U"], V=a[p]["V"] - b[p]["V"],
                             count=np.minimum(a[p]["count"], b[p]["count"]),
                             n_outside=a[p]["n_outside"])
                     for p in a})
            for (label, a), (_, b) in zip(rows_a, rows_b)]


def _key_length(mag):
    """A round number near the typical arrow, for the scale key."""
    return min([0.05, 0.1, 0.25, 0.5, 1.0, 2.0],
               key=lambda v: abs(v - np.nanpercentile(mag, 75)))


def plot_diff_planes_drift(res, rows, b_label, filename, a_label, arrow_note,
                           row_height=6.6):
    """The difference map of `plot_diff_planes`, with a measured drift field over it.

    `rows` is the output of `drift_rows`. The arrow magnification is computed per
    *column* and shared down it, so when faceting by generation the shrinking step size
    reads directly off the figure.
    """
    n_rows, n_cols = len(rows), len(res)
    fig, axes = plt.subplots(n_rows, n_cols, squeeze=False, facecolor=SURFACE,
                             figsize=(8.0 * n_cols, row_height * n_rows))

    # one magnification per plane, set by the longest row so no arrow overflows
    gain, keylen = {}, {}
    for p in res:
        mags = [np.hypot(f[p]["U"], f[p]["V"]) for _, f in rows]
        p90 = max(np.nanpercentile(m, 90) for m in mags if np.isfinite(m).any())
        cell_w = (res[p]["extent"][1] - res[p]["extent"][0]) / rows[0][1][p]["gx"].shape[1]
        gain[p] = (ARROW_FILL * cell_w / p90) if ARROW_GAIN == "auto" else ARROW_GAIN
        keylen[p] = _key_length(np.concatenate([m[np.isfinite(m)] for m in mags]))

    for row, (label, fields) in enumerate(rows):
        for col, (xk, yk) in enumerate(res):
            ax, r, f = axes[row, col], res[(xk, yk)], fields[(xk, yk)]
            dmax = np.abs(r["diff"]).max()
            im = ax.imshow(r["diff"], origin="lower", extent=r["extent"], cmap=DIVERGING,
                           vmin=-dmax, vmax=dmax, aspect="equal", zorder=0)
            ax.contour(r["gx"], r["gy"], r["d1"],
                       levels=[hdr_level(r["d1"], r["cell_area"])],
                       colors=GEN1_COLOR, linewidths=1.6, linestyles="--", zorder=3)
            ax.contour(r["gx"], r["gy"], r["d5"],
                       levels=[hdr_level(r["d5"], r["cell_area"])],
                       colors=GEN5_COLOR, linewidths=1.8, zorder=3)

            m = np.isfinite(f["U"])
            q = ax.quiver(f["gx"][m], f["gy"][m], f["U"][m], f["V"][m],
                          angles="xy", scale_units="xy", scale=1 / gain[(xk, yk)],
                          zorder=4, **ARROW_STYLE)

            title = f"{xk} / {yk}" + (f"   —   {label}" if label else "")
            _style(ax, xk, yk, title, r["extent"])
            ax.title.set_fontsize(13)
            ax.xaxis.label.set_fontsize(11)
            ax.yaxis.label.set_fontsize(11)

            cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.12)
            cb = fig.colorbar(im, cax=cax)
            cb.ax.tick_params(labelsize=8, colors=MUTED, length=2)
            cb.outline.set_visible(False)
            cb.set_label("Δ density", fontsize=10, color=MUTED)

            # one key per column, on the bottom row, since the scale is shared down it
            if row == n_rows - 1:
                k = keylen[(xk, yk)]
                ax.quiverkey(q, X=0.985, Y=-0.105, U=k, labelpos="W",
                             color=ARROW_STYLE["color"], labelcolor=MUTED,
                             label=f"{k:g} PC units / generation", labelsep=0.04,
                             fontproperties=dict(size=9))

    fig.legend(handles=[
        Line2D([], [], color=GEN1_COLOR, lw=1.6, ls="--",
               label=f"{a_label} — {BOUND_MASS:.0%} of vases"),
        Line2D([], [], color=GEN5_COLOR, lw=1.8,
               label=f"{b_label} — {BOUND_MASS:.0%} of vases"),
        Line2D([], [], color=ARROW_STYLE["color"], lw=1.2, marker=">", markersize=5,
               label=arrow_note)],
        loc="lower center", ncol=3, frameon=False, fontsize=10,
        labelcolor=INK, bbox_to_anchor=(0.5, 0.004))

    pad = 0.075 / n_rows
    fig.tight_layout(rect=(0, 2.2 * pad, 1, 1 - 0.1 / n_rows))
    fig.savefig(FIG_DIR / filename, dpi=200, facecolor=SURFACE)
    plt.show()


real_rows = drift_rows(real_steps)
null_rows = drift_rows(null_steps)
excess_rows = subtract_rows(real_rows, null_rows)
_tag = "_by_generation" if FACET_BY_GENERATION else ""

for (label, fields), (_, xfields) in zip(real_rows, excess_rows):
    for p in PLANES:
        f, x = fields[p], xfields[p]
        print(f"{p[0]}/{p[1]}{'  ' + label if label else ''}: "
              f"{int(np.isfinite(f['U']).sum())}/{f['U'].size} cells drawn  "
              f"median |drift| real={np.nanmedian(np.hypot(f['U'], f['V'])):.3f}  "
              f"real−null={np.nanmedian(np.hypot(x['U'], x['V'])):.3f} PC units/gen")

plot_diff_planes_drift(
    results, real_rows, "Real (human choice)", f"pc_density_diff_drift{_tag}.png",
    a_label="Null run (random choice)",
    arrow_note="mean champion movement per generation (real)")

plot_diff_planes_drift(
    results, excess_rows, "Real (human choice)", f"pc_density_diff_drift_vs_null{_tag}.png",
    a_label="Null run (random choice)",
    arrow_note="mean champion movement, real − null (attributable to choosing)")

## Final figure

One panel per plane. Three encodings, each answering a different question:

| Encoding | Question | Evidence behind it |
|---|---|---|
| **Fill** — blue `#2166AC` → white → red `#B2182B` | where does human choice concentrate gen-5 champions, relative to random choice? | 13,620 real vs 136,200 pooled null vases |
| **Black outline** | where is that difference more than noise? | within-lineage permutation test, FWER 5%, calibrated null-vs-null |
| **Arrows** | which way does choosing push a lineage, per generation? | per-trial drift removal (below) |

The 95% coverage contours are gone: with the null baseline both runs occupy nearly the
same region, so those lines carried little and competed with the significance outline.

### Accounting for drift: why this is not a subtraction of two vector fields

Subtracting the null field from the real field is reasonable, and it is what
`DRIFT_MODE = "subtract"` still does. But your instinct was right that it is awkward —
both fields are estimated with their own sampling error, and the difference of two noisy
vectors can point somewhere neither of them does, especially in thinly-populated cells
where the errors are largest.

There is an exact alternative, and it needs no null run at all. From generation 2 on,
every trial offers the incumbent champion and one novel child. A coin-flip chooser moves
the lineage by `(child − parent)/2` in expectation. So for each individual step, the part
of the observed move attributable to the person is

```
(1{chose the child} − ½) · (child − parent)
```

evaluated at the parent's position. The generative step `(child − parent)` is **the same
single draw in both terms**, so drift cancels exactly, per observation, rather than on
average across two separately-estimated fields. The gain is threefold: no cross-run
estimation error, far lower variance, and no cells lost to requiring a minimum count in
*both* runs.

The null run then becomes the validation rather than the correction — applying the same
estimator to it must return ~zero everywhere, since its choices are random. The cell
prints that ratio.

### On the log scale

Left linear, deliberately. PC scores are signed and centred on zero, so `log` is
undefined over half the range; `symlog` would work but introduces an arbitrary threshold.
More importantly it would not smooth anything — smoothness is set by the KDE bandwidth,
not the axis — and the plotted quantity is a density *per unit PC area*, so on a
nonlinear axis equal screen areas would stop representing equal probability mass unless
divided through by the Jacobian. Contour shapes and the apparent size of the significant
regions would distort. If the landscape looks rough, raise the bandwidth in
`compute_planes`.

### Output

SVG plus a matching PNG, both at 600 dpi. Axes, arrows, contours and text are true
vectors in the SVG; the density layer is an image by nature and embeds as a raster, which
is what the dpi setting controls.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# FINAL FIGURES — one per comparison, identical styling
#   fill    Δ density (real − null)      blue #2166AC · white · red #B2182B
#   outline regions significant at FWER 5% (permutation test)
#   arrows  movement attributable to choosing (per-trial drift removal)
# ═════════════════════════════════════════════════════════════════════════════
from matplotlib.colors import SymLogNorm

# ── Settings ────────────────────────────────────────────────────────────────
FINAL_CMAP = LinearSegmentedColormap.from_list(
    "final_bwr", ["#2166AC", "#FFFFFF", "#B2182B"])   # low · neutral · high
FINAL_SIG_COLOUR = "#000000"
FINAL_SIG_LW = 1.5
FINAL_DPI = 600
FINAL_STEM = "pc_density_diff_final"

FINAL_NORM = "symlog"   # "symlog" — log beyond the significance threshold,
                        #            linear (and near-white) inside it
                        # "linear" — plain symmetric scale
FINAL_LINSCALE = 0.55   # how much colour the linear core is given

DRIFT_MODE = "paired"   # "paired"  — per-trial drift removal (recommended)
                        # "subtract" — real field minus null field

FINAL_CHOICE_ARROWS = "lineage"
# "lineage"  — the within-round 2AFC selection field, identical to the gen-5
#              figure. Dense (54,480 steps) and well estimated.
# "crowning" — the final-choice step itself: crowned champion minus the mean of
#              that participant's five round champions, anchored at that mean.
#              Describes the step the figure is actually about, but rests on
#              2,724 vectors, so it needs a coarser grid.
CROWN_ARROW_N = 10
CROWN_MIN_COUNT = 15
# ─────────────────────────────────────────────────────────────────────────────


def selection_steps(df, keys):
    """Champion steps carrying only the displacement attributable to CHOOSING.

    From generation 2 on, every trial offers the incumbent champion
    (`is_parent_vase`) and one novel child. A coin-flip chooser moves the
    lineage by (child − parent)/2 in expectation, so the part of the observed
    move that is down to the person is

        (1{chose the child} − ½) · (child − parent)  =  ½ · (chosen − rejected)

    evaluated at the parent's position. The generative step (child − parent) is
    the same single draw in both terms, so drift cancels exactly per
    observation rather than on average across two estimated fields.

    Returned in the column layout `drift_field` expects (`PCn` = where the step
    starts, `PCn_next` = start + displacement), so every arrow toggle applies.
    """
    idx = keys + ["generation_key", "vase"]
    wide = (df.pivot_table(index=idx, columns="pc_name", values="pc_value")
              .reset_index())
    flags = df.drop_duplicates(idx)[idx + ["is_parent_vase", "is_selected_vase"]]
    wide = wide.merge(flags, on=idx)
    wide["gen"] = wide["generation_key"].str.extract(r"gen_(\d+)").astype(int)
    wide = wide[wide["gen"] >= 2]          # generation 1 has no incumbent

    on = keys + ["gen"]
    par = wide[wide["is_parent_vase"]].set_index(on).sort_index()
    nov = wide[~wide["is_parent_vase"]].set_index(on).sort_index()
    common = par.index.intersection(nov.index)
    par, nov = par.loc[common], nov.loc[common]
    assert (par["is_selected_vase"].to_numpy()
            != nov["is_selected_vase"].to_numpy()).all(), \
        "a trial did not have exactly one winner"

    chose_child = nov["is_selected_vase"].to_numpy().astype(float)
    out = par[[]].copy()
    for pc in [c for p in PLANES for c in p]:
        p0, c0 = par[pc].to_numpy(), nov[pc].to_numpy()
        out[pc] = p0
        out[f"{pc}_next"] = p0 + (chose_child - 0.5) * (c0 - p0)
    return out.reset_index()


def crowning_steps(champs, crowned, keys):
    """Selection displacement of the final-choice ('king of kings') step.

    The counterfactual here is a 5-way coin flip: picking one of the round
    champions at random gives their MEAN in expectation, so the part
    attributable to choosing is (crowned − mean of the five), anchored at that
    mean. Same estimator family as `selection_steps` — for a k-way choice the
    drift-free residual is always `chosen − mean(candidates)`, which reduces to
    ½(chosen − rejected) when k = 2.
    """
    pcs = [c for p in PLANES for c in p]
    m = champs.groupby(keys)[pcs].mean()
    c = crowned.set_index(keys)[pcs]
    common = m.index.intersection(c.index)
    m, c = m.loc[common], c.loc[common]
    out = m[[]].copy()
    for pc in pcs:
        out[pc] = m[pc].to_numpy()
        out[f"{pc}_next"] = c[pc].to_numpy()
    out["gen"] = 6            # a single pseudo-step, so faceting leaves it whole
    return out.reset_index()


real_sel = selection_steps(pc_clean, REAL_KEYS)
null_sel = selection_steps(null_data, NULL_KEYS)

# The same estimator on the null run must return ~zero: its choices are random,
# so there is no selection to find. This is the check that the correction works.
_vr, _vn = [], []
for xk, yk in PLANES:
    for src, acc in ((real_sel, _vr), (null_sel, _vn)):
        f = drift_field(src, xk, yk, results[(xk, yk)]["extent"])
        acc.append(np.nanmedian(np.hypot(f["U"], f["V"])))
print(f"median |selection displacement|  real {np.mean(_vr):.4f}  "
      f"null {np.mean(_vn):.4f} PC units/gen   "
      f"(null should be ~0; ratio {np.mean(_vr) / max(np.mean(_vn), 1e-9):.0f}x)")


def rows_for(res, steps, facet=None, **kw):
    """drift_rows, but on an arbitrary result set's window rather than `results`."""
    facet = FACET_BY_GENERATION if facet is None else facet
    if not facet:
        return [(None, {p: drift_field(steps, *p, res[p]["extent"], **kw)
                        for p in PLANES})]
    return [(f"gen {g} → {g + 1}",
             {p: drift_field(steps, *p, res[p]["extent"], gen=g, **kw) for p in PLANES})
            for g in sorted(steps["gen"].unique())]


def plot_final(res, perm, rows, filename, arrow_note, row_height=6.6):
    n_rows, n_cols = len(rows), len(res)
    fig, axes = plt.subplots(n_rows, n_cols, squeeze=False, facecolor=SURFACE,
                             figsize=(8.0 * n_cols, row_height * n_rows))

    gain, keylen = {}, {}
    for p in res:
        mags = [np.hypot(f[p]["U"], f[p]["V"]) for _, f in rows]
        finite = [m for m in mags if np.isfinite(m).any()]
        p90 = max(np.nanpercentile(m, 90) for m in finite)
        cell_w = (res[p]["extent"][1] - res[p]["extent"][0]) / rows[0][1][p]["gx"].shape[1]
        gain[p] = (ARROW_FILL * cell_w / p90) if ARROW_GAIN == "auto" else ARROW_GAIN
        keylen[p] = _key_length(np.concatenate([m[np.isfinite(m)] for m in finite]))

    for row, (label, fields) in enumerate(rows):
        for col, (xk, yk) in enumerate(res):
            ax, r, f = axes[row, col], res[(xk, yk)], fields[(xk, yk)]
            dmax = np.abs(r["diff"]).max()

            # Tie the colour scale to the statistics: inside ±threshold the
            # difference is not distinguishable from noise, so that band gets a
            # flat near-white linear core and everything beyond it is log.
            lt = float(np.clip(perm[(xk, yk)]["thresh"], dmax * 1e-4, dmax * 0.9))
            norm = (SymLogNorm(linthresh=lt, linscale=FINAL_LINSCALE,
                               vmin=-dmax, vmax=dmax, base=10)
                    if FINAL_NORM == "symlog" else Normalize(-dmax, dmax))

            im = ax.imshow(r["diff"], origin="lower", extent=r["extent"],
                           cmap=FINAL_CMAP, norm=norm, aspect="equal",
                           zorder=0, interpolation="bilinear")

            # Regions beyond the simultaneous FWER-5% threshold — the
            # permutation result, not an eyeballed level.
            ax.contour(r["gx"], r["gy"], perm[(xk, yk)]["sig"].astype(float),
                       levels=[0.5], colors=FINAL_SIG_COLOUR,
                       linewidths=FINAL_SIG_LW, zorder=3)

            m = np.isfinite(f["U"])
            q = ax.quiver(f["gx"][m], f["gy"][m], f["U"][m], f["V"][m],
                          angles="xy", scale_units="xy",
                          scale=1 / gain[(xk, yk)], zorder=4, **ARROW_STYLE)

            title = f"{xk} / {yk}" + (f"   —   {label}" if label else "")
            _style(ax, xk, yk, title, r["extent"])
            ax.title.set_fontsize(13)
            ax.xaxis.label.set_fontsize(11)
            ax.yaxis.label.set_fontsize(11)

            cax = make_axes_locatable(ax).append_axes("right", size="4%", pad=0.12)
            cb = fig.colorbar(im, cax=cax)
            if FINAL_NORM == "symlog":
                cb.set_ticks([-dmax, -lt, 0.0, lt, dmax])
                cb.set_ticklabels([f"{-dmax:.3g}", f"−thr", "0", f"+thr", f"{dmax:.3g}"])
            cb.ax.tick_params(labelsize=8, colors=MUTED, length=2)
            cb.outline.set_visible(False)
            cb.set_label("Δ density  (real − null)", fontsize=10, color=MUTED)

            if row == n_rows - 1:
                k = keylen[(xk, yk)]
                ax.quiverkey(q, X=0.985, Y=-0.105, U=k, labelpos="W",
                             color=ARROW_STYLE["color"], labelcolor=MUTED,
                             label=f"{k:g} PC units / generation", labelsep=0.04,
                             fontproperties=dict(size=9))

    handles = [
        Line2D([], [], color=FINAL_SIG_COLOUR, lw=FINAL_SIG_LW,
               label=f"significant at FWER {ALPHA:.0%} "
                     f"({N_PERM:,} within-lineage permutations)"),
        Line2D([], [], color=ARROW_STYLE["color"], lw=1.2, marker=">",
               markersize=5, label=arrow_note)]
    if FINAL_NORM == "symlog":
        handles.append(Line2D([], [], color="none",
                              label="colour: symlog, linear inside ±threshold"))
    fig.legend(handles=handles, loc="lower center", ncol=len(handles),
               frameon=False, fontsize=10, labelcolor=INK,
               bbox_to_anchor=(0.5, 0.004))

    pad = 0.075 / n_rows
    fig.tight_layout(rect=(0, 2.2 * pad, 1, 1 - 0.1 / n_rows))
    # SVG keeps axes, arrows, contours and text as vectors; the density layer is
    # an image by nature and embeds as a raster, which is what dpi controls.
    for ext in ("svg", "png"):
        fig.savefig(FIG_DIR / f"{filename}.{ext}", dpi=FINAL_DPI,
                    facecolor=SURFACE, bbox_inches="tight")
    plt.show()


_tag = "_by_generation" if FACET_BY_GENERATION else ""
_note = ("mean movement attributable to choosing (per-trial drift removal)"
         if DRIFT_MODE == "paired" else "mean champion movement, real − null")

# ---- figure 1: gen-5 champions ---------------------------------------------
rows_gen5 = (rows_for(results, real_sel) if DRIFT_MODE == "paired"
             else subtract_rows(drift_rows(real_steps), drift_rows(null_steps)))
plot_final(results, perm_gen5, rows_gen5,
           f"{FINAL_STEM}_gen5{_tag}", _note)

# ---- figure 2: final choice, same styling ----------------------------------
if FINAL_CHOICE_ARROWS == "crowning":
    real_crown = crowning_steps(gen5_select, final_choice, ["participant_id"])
    null_crown = crowning_steps(null_gen5_select, null_final_choice,
                                ["null_repeat", "participant_id"])
    rows_fc = rows_for(results_final, real_crown, facet=False,
                       n=CROWN_ARROW_N, min_count=CROWN_MIN_COUNT)
    _n_drawn = sum(int(np.isfinite(f["U"]).sum()) for f in rows_fc[0][1].values())
    print(f"crowning field: {len(real_crown):,} vectors, {_n_drawn} cells drawn "
          f"(raise CROWN_ARROW_N or lower CROWN_MIN_COUNT if that is too sparse)")
    _note_fc = "crowned champion − mean of that participant's five"
else:
    rows_fc = rows_for(results_final, real_sel)
    _note_fc = _note

plot_final(results_final, perm_final, rows_fc,
           f"{FINAL_STEM}_final_choice{_tag}", _note_fc)
print(f"saved {FINAL_STEM}_gen5{_tag}.svg and {FINAL_STEM}_final_choice{_tag}.svg "
      f"at {FINAL_DPI} dpi")
